# **Predicting Used Car Selling Price Using Machine Learning**

Team members:

1.   Heng-Jui Chu
2.   Simbarashe Mpofu
3.   Simbarashe Simbangegavi

## **Dataset and Github**


**Dataset**

[Used Car Price Dataset (Kaggle)](https://www.kaggle.com/datasets/therohithanand/used-car-price-prediction)

**Github**

[Machine Learning Project GitHub](https://github.com/Simba-gwu/Machine_learning_project)

## **Introduction**


Nowadays, many people use online platforms to buy and sell used cars, and pricing is always an important issue. The price of a used car can be affected by many factors, such as brand, mileage, age, and condition. Some websites, like CarMax, provide estimated prices, but we usually do not know how accurate these estimates are or how they are calculated. Therefore, we wonder: can a machine learning model predict used car prices more accurately than existing online estimation tools?

The motivation for this project comes from the need for more reliable and transparent pricing. Buyers want to ensure they are not paying too much, while sellers want to set a reasonable price to attract buyers. However, current online tools do not clearly explain their prediction processes, which makes it difficult to fully trust them. Therefore, building a machine learning model based on real data can be a better way to understand how prices are determined.

In this project, we will use a used car dataset to train a machine learning model and evaluate its performance. The main questions we want to answer are:

1. How well can a machine learning model predict used car prices?
2. What factors have the biggest impact on car prices?

Additionally, this project aims to provide a better understanding of used car pricing and demonstrate how machine learning can be used as a useful tool for price prediction.

## **Experiment**

In [1]:
from google.colab import drive
import sys

# Mount Google Drive
drive.mount('/content/drive')

# Get the absolute path of the current folder
abspath_curr = '/content/drive/My Drive/Colab Notebooks/ML project/'

Mounted at /content/drive


In [2]:
import os
from pathlib import Path



In [3]:
print(os.listdir(abspath_curr))


['vehicles.csv', 'used_car_dataset.csv', 'result', 'Used_Car_Price_Prediction.ipynb']


**Warning**

In [4]:
import warnings

# Ignore warnings
warnings.filterwarnings('ignore')

**Matplotlib**

In [5]:
import matplotlib.pyplot as plt
%matplotlib inline

# Set matplotlib sizes
plt.rc('font', size=20)
plt.rc('axes', titlesize=20)
plt.rc('axes', labelsize=20)
plt.rc('xtick', labelsize=20)
plt.rc('ytick', labelsize=20)
plt.rc('legend', fontsize=20)
plt.rc('figure', titlesize=20)

**TensorFlow**

In [6]:
# The magic below allows us to use tensorflow version 2.x
%tensorflow_version 2.x
import tensorflow as tf
from tensorflow import keras

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.


**Random seed**

In [7]:
# The random seed
random_seed = 42

# Set random seed in tensorflow
tf.random.set_seed(random_seed)

# Set random seed in numpy
import numpy as np
np.random.seed(random_seed)

## **Import tools**

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import shap


**Data Preprocessing**

**Loading the data**

In [9]:
df_raw_train = pd.read_csv(abspath_curr + '/vehicles.csv',
                           header=0)

# Make a copy of df_raw_train
df_train = df_raw_train.copy(deep=True)

# Get the name of the target
target = 'price'


In [10]:
# Print the dimension of df_train
pd.DataFrame([[df_train.shape[0], df_train.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,426880,26


In [11]:
# Print the first 5 rows of df_train
df_train.head()

,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,...,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
0,7222695916,https://prescott.craigslist.org/cto/d/prescott...,prescott,https://prescott.craigslist.org,6000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaN
1,7218891961,https://fayar.craigslist.org/ctd/d/bentonville...,fayetteville,https://fayar.craigslist.org,11900,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaN
2,7221797935,https://keys.craigslist.org/cto/d/summerland-k...,florida keys,https://keys.craigslist.org,21000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN,NaN
3,7222270760,https://worcester.craigslist.org/cto/d/west-br...,worcester / central MA,https://worcester.craigslist.org,1500,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN,NaN,NaN
4,7210384030,https://greensboro.craigslist.org/cto/d/trinit...,greensboro,https://greensboro.craigslist.org,4900,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN,NaN,NaN


Removing unwanted columns

In [12]:
# Create a list of all columns that won't help with price prediction
cols_to_drop = [
     'url', 'region_url', 'VIN', 'image_url',
    'description', 'county','size', 'lat', 'long'
]

# Drop them all at once
df_train = df_train.drop(columns=cols_to_drop)

# Double-check the remaining columns
print(df_train.columns)

Index(['id', 'region', 'price', 'year', 'manufacturer', 'model', 'condition',
       'cylinders', 'fuel', 'odometer', 'title_status', 'transmission',
       'drive', 'type', 'paint_color', 'state', 'posting_date'],
      dtype='object')


In [13]:
# Print the first 5 rows of df_train
df_train.head()

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state,posting_date
0,7222695916,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN
1,7218891961,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN
2,7221797935,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN
3,7222270760,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN
4,7210384030,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN


**Splitting the data**

The code below shows how to divide the training data into training (60%) ,validation (20%) and testing (20%).

In [14]:
from sklearn.model_selection import train_test_split

# Divide the training data into training (60%) , temp (40%)
df_train, df_temp = train_test_split(df_train, train_size=0.6, random_state=random_seed)

# Second split: split temp into validation and test (20% each)
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=random_seed)

# Reset the index
df_train, df_val , df_test = df_train.reset_index(drop=True), df_val.reset_index(drop=True) , df_test.reset_index(drop=True)

In [15]:
# Print the dimension of df_train
pd.DataFrame([[df_train.shape[0], df_train.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,256128,17


In [16]:
# Print the dimension of df_val
pd.DataFrame([[df_val.shape[0], df_val.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,85376,17


In [17]:
# Print the dimension of df_test
pd.DataFrame([[df_test.shape[0], df_test.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,85376,17


**Identifying uncommon variables**

The code below shows how to find common variables between the training, validation and test data.

In [18]:
def common_var_checker(df_train, df_val, df_test, target):
    """
    The common variables checker

    Parameters
    ----------
    df_train : the dataframe of training data
    df_val : the dataframe of validation data
    df_test : the dataframe of test data
    target : the name of the target

    Returns
    ----------
    The dataframe of common variables between the training, validation and test data
    """

    # Get the dataframe of common variables between the training, validation and test data
    df_common_var = pd.DataFrame(np.intersect1d(np.intersect1d(df_train.columns, df_val.columns), np.union1d(df_test.columns, [target])),
                                 columns=['common var'])

    return df_common_var

In [19]:
# Call common_var_checker
#
df_common_var = common_var_checker(df_train, df_val, df_test, target)

# Print df_common_var
df_common_var

,common var
0,condition
1,cylinders
2,drive
3,fuel
4,id
5,manufacturer
6,model
7,odometer
8,paint_color
9,posting_date


The code below shows how to find features in the training data but not in the validation or test data.

In [20]:
# Get the features in the training data but not in the validation or test data
uncommon_feature_train_not_val_test = np.setdiff1d(df_train.columns, df_common_var['common var'])

# Print the uncommon features
pd.DataFrame(uncommon_feature_train_not_val_test, columns=['uncommon feature'])

,uncommon feature


The code below shows how to find the features in the validation data but not in the training or test data.

In [21]:
# Get the features in the validation data but not in the training or test data
uncommon_feature_val_not_train_test = np.setdiff1d(df_val.columns, df_common_var['common var'])

# Print the uncommon features
pd.DataFrame(uncommon_feature_val_not_train_test, columns=['uncommon feature'])

,uncommon feature


The code below shows how to find the features in the test data but not in the training or validation data.

In [22]:
# Get the features in the test data but not in the training or validation data
uncommon_feature_test_not_train_val = np.setdiff1d(df_test.columns, df_common_var['common var'])

# Print the uncommon features
pd.DataFrame(uncommon_feature_test_not_train_val, columns=['uncommon feature'])

,uncommon feature


**Removing uncommon features**

In [23]:
# Remove the uncommon features from the training data
df_train = df_train.drop(columns=uncommon_feature_train_not_val_test)

# Print the first 5 rows of df_train
df_train.head()

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state,posting_date
0,7313343872,orange county,11595,2014.0,kia,optima,NaN,4 cylinders,gas,115166.0,clean,automatic,fwd,sedan,custom,ca,2021-04-27T08:42:29-0700
1,7304860433,daytona beach,3100,2004.0,infiniti,q45,excellent,8 cylinders,gas,202.0,rebuilt,automatic,rwd,sedan,NaN,fl,2021-04-10T16:05:37-0400
2,7315582037,san diego,0,2019.0,toyota,4runner sr5 suv,excellent,NaN,gas,20065.0,clean,automatic,NaN,SUV,NaN,ca,2021-05-01T17:08:55-0700
3,7316009083,billings,12900,2018.0,chevrolet,malibu,excellent,4 cylinders,gas,93000.0,clean,automatic,fwd,sedan,yellow,mt,2021-05-02T18:21:27-0600
4,7314300377,tampa bay area,22790,2012.0,ford,f-150 xlt,excellent,6 cylinders,gas,98549.0,clean,automatic,4wd,truck,NaN,fl,2021-04-29T10:38:02-0400


In [24]:
# Remove the uncommon features from the validation data
df_val = df_val.drop(columns=uncommon_feature_val_not_train_test)

# Print the first 5 rows of df_val
df_val.head()

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state,posting_date
0,7310049518,raleigh / durham / CH,5950,2010.0,honda,odyssey,good,6 cylinders,gas,184713.0,clean,automatic,fwd,van,blue,nc,2021-04-20T17:32:06-0400
1,7313947988,boise,8495,2011.0,dodge,grand caravan,good,6 cylinders,gas,94164.0,clean,automatic,fwd,mini-van,blue,id,2021-04-28T12:22:39-0600
2,7316808274,denver,27990,2020.0,ford,transit connect cargo van,good,NaN,other,3772.0,clean,other,fwd,van,white,co,2021-05-04T11:21:16-0600
3,7310254946,southeast KS,41000,2019.0,mini,t camper van,excellent,4 cylinders,gas,29288.0,clean,automatic,NaN,van,NaN,ks,2021-04-21T07:35:19-0500
4,7315029138,san antonio,16995,2012.0,ford,expedition,NaN,NaN,other,92248.0,clean,automatic,fwd,SUV,NaN,tx,2021-04-30T16:15:35-0500


In [25]:
# Remove the uncommon features from the test data
df_test = df_test.drop(columns=uncommon_feature_test_not_train_val)

# Print the first 5 rows of df_test
df_test.head()

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state,posting_date
0,7310406480,south jersey,16985,2015.0,lincoln,mkc,NaN,NaN,gas,85933.0,clean,automatic,4wd,SUV,NaN,nj,2021-04-21T13:06:01-0400
1,7306331544,cleveland,500,2011.0,jeep,grand cherokee,NaN,NaN,gas,80125.0,clean,automatic,NaN,NaN,NaN,oh,2021-04-13T15:48:11-0400
2,7311059803,gold country,1800,1988.0,toyota,pickup,fair,4 cylinders,gas,298000.0,clean,manual,rwd,NaN,brown,ca,2021-04-22T14:10:03-0700
3,7314759305,lynchburg,26990,2014.0,chevrolet,silverado 1500 crew,good,6 cylinders,gas,63129.0,clean,automatic,NaN,pickup,black,va,2021-04-30T09:31:18-0400
4,7307655476,wyoming,29999,2013.0,gmc,sierra,NaN,NaN,diesel,229023.0,clean,automatic,4wd,pickup,NaN,wy,2021-04-16T07:37:36-0600


**Handling identifiers**

**Combining the training, validation and test data**

The code below shows how to combine the training, validation and test da

In [26]:
# Combine df_train, df_val and df_test
df = pd.concat([df_train, df_val, df_test], sort=False)

**Identifying identifiers**

The code below shows how to find identifiers from data.

In [27]:
def id_checker(df, dtype='float'):
    """
    The identifier checker

    Parameters
    ----------
    df : dataframe
    dtype : the data type identifiers cannot have, 'float' by default
            i.e., if a feature has this data type, it cannot be an identifier

    Returns
    ----------
    The dataframe of identifiers
    """

    # Get the dataframe of identifiers
    df_id = df[[var for var in df.columns
                # If the data type is not dtype
                if (df[var].dtype != dtype
                    # If the value is unique for each sample
                    and df[var].nunique(dropna=True) == df[var].notnull().sum())]]

    return df_id

In [28]:
# Call id_checker on df
# See the implementation in pmlm_utilities.ipynb
df_id = id_checker(df)

# Print the first 5 rows of df_id
df_id.head()

,id
0,7313343872
1,7304860433
2,7315582037
3,7316009083
4,7314300377


**Removing identifiers**

The code below shows how to remove identifiers from data.

In [30]:
import numpy as np

# Remove identifiers from df_train
df_train.drop(columns=np.intersect1d(df_id.columns, df_train.columns), inplace=True)

# Remove identifiers from df_val
df_val.drop(columns=np.intersect1d(df_id.columns, df_val.columns), inplace=True)

# Remove identifiers from df_test
df_test.drop(columns=np.intersect1d(df_id.columns, df_test.columns), inplace=True)

**Handling missing data**

**Combining the training, validation and test data**

The code below shows how to combine the training, validation and test data.

In [31]:
# Combine df_train, df_val and df_test
df = pd.concat([df_train, df_val, df_test], sort=False)

**Identifying missing values**

The code below shows how to find variables with NaN, their proportion of NaN and data type.

In [32]:
def nan_checker(df):
    """
    The NaN checker

    Parameters
    ----------
    df : the dataframe

    Returns
    ----------
    The dataframe of variables with NaN, their proportion of NaN and data type
    """

    # Get the dataframe of variables with NaN, their proportion of NaN and data type
    df_nan = pd.DataFrame([[var, df[var].isna().sum() / df.shape[0], df[var].dtype]
                           for var in df.columns if df[var].isna().sum() > 0],
                          columns=['var', 'proportion', 'dtype'])

    # Sort df_nan in accending order of the proportion of NaN
    df_nan = df_nan.sort_values(by='proportion', ascending=False).reset_index(drop=True)

    return df_nan

In [33]:
# Call nan_checker on df
# See the implementation in pmlm_utilities.ipynb
df_nan = nan_checker(df)

# Print df_nan
df_nan

,var,proportion,dtype
0,cylinders,0.416225,object
1,condition,0.407852,object
2,drive,0.305863,object
3,paint_color,0.305011,object
4,type,0.217527,object
5,manufacturer,0.041337,object
6,title_status,0.019308,object
7,model,0.012362,object
8,odometer,0.010307,float64
9,fuel,0.007058,object


In [34]:
# Print the unique data type of variables with NaN
pd.DataFrame(df_nan['dtype'].unique(), columns=['dtype'])

,dtype
0,object
1,float64


In [35]:
df_miss = df_nan[df_nan['dtype'] == 'object'].reset_index(drop=True)
df_miss

,var,proportion,dtype
0,cylinders,0.416225,object
1,condition,0.407852,object
2,drive,0.305863,object
3,paint_color,0.305011,object
4,type,0.217527,object
5,manufacturer,0.041337,object
6,title_status,0.019308,object
7,model,0.012362,object
8,fuel,0.007058,object
9,transmission,0.005988,object


In [36]:
df_miss_num = df_nan[df_nan['dtype'] == 'float64'].reset_index(drop=True)
df_miss_num

,var,proportion,dtype
0,odometer,0.010307,float64
1,year,0.002823,float64


**Separating the training, validation and test data**

The code below shows how to separate the training, validation and test data.


In [37]:
#Separating the training data
df_train = df.iloc[:df_train.shape[0], :]

# Separating the validation data
df_val = df.iloc[df_train.shape[0]:df_train.shape[0] + df_val.shape[0], :]

# Separating the test data
df_test = df.iloc[df_train.shape[0] + df_val.shape[0]:, :]

In [38]:
# Print the dimension of df_train
pd.DataFrame([[df_train.shape[0], df_train.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,256128,16


In [39]:
# Print the dimension of df_val
pd.DataFrame([[df_val.shape[0], df_val.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,85376,16


In [40]:
# Print the dimension of df_test
pd.DataFrame([[df_test.shape[0], df_test.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,85376,16


In [41]:
# Print the dimension of df_train
pd.DataFrame([[df_train.shape[0], df_train.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,256128,16


Handling missing values in Cylinders by changing it to a float and imouting the missing with median

In [42]:
# --- 1. HANDLE CYLINDERS (Numerical + Flag) ---
# Extract digits only (e.g., "8 cylinders" -> 8.0)
df_train['cylinders'] = df_train['cylinders'].str.extract('(\d+)').astype(float)
df_val['cylinders'] = df_val['cylinders'].str.extract('(\d+)').astype(float)
df_test['cylinders'] = df_test['cylinders'].str.extract('(\d+)').astype(float)

# Create the 'unknown' flag BEFORE filling the NaNs
# This preserves the "signal" that the seller didn't provide info
df_train['cylinders_unknown'] = df_train['cylinders'].isnull().astype(int)
df_val['cylinders_unknown'] = df_val['cylinders'].isnull().astype(int)
df_test['cylinders_unknown'] = df_test['cylinders'].isnull().astype(int)

# Fill the NaNs with the median (usually 6.0)
cyl_median_train = df_train['cylinders'].median()
cyl_median_val = df_val['cylinders'].median()
cyl_median_test = df_test['cylinders'].median()
df_train['cylinders'] = df_train['cylinders'].fillna(cyl_median_train)
df_val['cylinders'] = df_train['cylinders'].fillna(cyl_median_val)
df_test['cylinders'] = df_train['cylinders'].fillna(cyl_median_test)

Hnadling missing values by dropping rows from columns with few percentage of missing values

In [43]:
vars_to_drop_rows = ['model', 'fuel', 'transmission', 'posting_date','cylinders']

# 2. Apply your logic using inplace=False
if len(vars_to_drop_rows) > 0:
    # We must re-assign the variable because inplace=False returns a NEW dataframe
    df_train = df_train.dropna(subset=np.intersect1d(vars_to_drop_rows, df_train.columns),
                               inplace=False)

    df_val = df_val.dropna(subset=np.intersect1d(vars_to_drop_rows, df_val.columns),
                             inplace=False)

    df_test = df_test.dropna(subset=np.intersect1d(vars_to_drop_rows, df_test.columns),
                              inplace=False)

# 3. Quick check of the new row count
print(f"Cleaned Train rows: {df_train.shape[0]}")

Cleaned Train rows: 250116


In [44]:
# Print the dimension of df_train
pd.DataFrame([[df_train.shape[0], df_train.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,250116,17


In [45]:
df = pd.concat([df_train, df_val, df_test], sort=False)

In [46]:
# Call nan_checker on df
# See the implementation in pmlm_utilities.ipynb
df_nan = nan_checker(df)

# Print df_nan
df_nan

,var,proportion,dtype
0,condition,0.399115,object
1,drive,0.303101,object
2,paint_color,0.301021,object
3,type,0.214431,object
4,manufacturer,0.040820,object
5,title_status,0.016762,object
6,odometer,0.008438,float64
7,year,0.002248,float64


In [47]:
df_miss = df_nan[df_nan['dtype'] == 'object'].reset_index(drop=True)
df_miss

,var,proportion,dtype
0,condition,0.399115,object
1,drive,0.303101,object
2,paint_color,0.301021,object
3,type,0.214431,object
4,manufacturer,0.040820,object
5,title_status,0.016762,object


In [48]:
#Separating the training data
df_train = df.iloc[:df_train.shape[0], :]

# Separating the validation data
df_val = df.iloc[df_train.shape[0]:df_train.shape[0] + df_val.shape[0], :]

# Separating the test data
df_test = df.iloc[df_train.shape[0] + df_val.shape[0]:, :]

**Handling Missing Values by Adding an "Unknown" Category**

In [49]:
from sklearn.impute import SimpleImputer

si = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='Unknown')

df_train[df_miss['var']] = si.fit_transform(df_train[df_miss['var']])
df_val[df_miss['var']] = si.transform(df_val[df_miss['var']])
df_test[df_miss['var']] = si.transform(df_test[df_miss['var']])

Imputing missing values

The code below shows how to use the median of a numerical feature to impute its missing values.

In [50]:
from sklearn.impute import SimpleImputer

# If there are missing values
if len(df_miss_num['var']) > 0:
    # The SimpleImputer
    si = SimpleImputer(missing_values=np.nan, strategy='median')

    # Impute the variables with missing values in df_train, df_val and df_test
    df_train[df_miss_num['var']] = si.fit_transform(df_train[df_miss_num['var']])
    df_val[df_miss_num['var']] = si.transform(df_val[df_miss_num['var']])
    df_test[df_miss_num['var']] = si.transform(df_test[df_miss_num['var']])

Handling date time variables

Transforming date time variables The code below shows how to transform date time variables into the following 6 datetime types:

year

month

day

hour

minute

second

In [51]:
# Get the date time variables
datetime_vars = ['posting_date']

In [52]:
print(df_train['posting_date'].dtype)
print(df_train['posting_date'].head())

object
0    2021-04-27T08:42:29-0700
1    2021-04-10T16:05:37-0400
2    2021-05-01T17:08:55-0700
3    2021-05-02T18:21:27-0600
4    2021-04-29T10:38:02-0400
Name: posting_date, dtype: object


In [53]:
def datetime_transformer(df, datetime_vars):
    """
    The datetime transformer

    Parameters
    ----------
    df : the dataframe
    datetime_vars : the datetime variables

    Returns
    ----------
    The dataframe where datetime_vars are transformed into the following 6 datetime types:
    year, month, day, hour, minute and second
    """

    # The dictionary with key as datetime type and value as datetime type operator
    dict_ = {'year'   : lambda x : x.dt.year,
             'month'  : lambda x : x.dt.month,
             }

    # Make a copy of df
    df_datetime = df.copy(deep=True)

    # For each variable in datetime_vars
    for var in datetime_vars:
        # Cast the variable to datetime
        df_datetime[var] = pd.to_datetime(df_datetime[var], errors='coerce', utc=True)

        # For each item (datetime_type and datetime_type_operator) in dict_
        for datetime_type, datetime_type_operator in dict_.items():
            # Add a new variable to df_datetime where:
            # the variable's name is var + '_' + datetime_type
            # the variable's values are the ones obtained by datetime_type_operator
            df_datetime[var + '_' + datetime_type] = datetime_type_operator(df_datetime[var])

    # Remove datetime_vars from df_datetime
    df_datetime = df_datetime.drop(columns=datetime_vars)

    return df_datetime

In [54]:
# Call datetime_transformer on df_train
# See the implementation in pmlm_utilities.ipynb
df_train = datetime_transformer(df_train, datetime_vars)

# Print the first 5 rows of df_train
df_train.head()

,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state,cylinders_unknown,posting_date_year,posting_date_month
0,orange county,11595,2014.0,kia,optima,Unknown,4.0,gas,115166.0,clean,automatic,fwd,sedan,custom,ca,0,2021,4
1,daytona beach,3100,2004.0,infiniti,q45,excellent,8.0,gas,202.0,rebuilt,automatic,rwd,sedan,Unknown,fl,0,2021,4
2,san diego,0,2019.0,toyota,4runner sr5 suv,excellent,6.0,gas,20065.0,clean,automatic,Unknown,SUV,Unknown,ca,1,2021,5
3,billings,12900,2018.0,chevrolet,malibu,excellent,4.0,gas,93000.0,clean,automatic,fwd,sedan,yellow,mt,0,2021,5
4,tampa bay area,22790,2012.0,ford,f-150 xlt,excellent,6.0,gas,98549.0,clean,automatic,4wd,truck,Unknown,fl,0,2021,4


In [55]:
# Call datetime_transformer on df_val
# See the implementation in pmlm_utilities.ipynb
df_val = datetime_transformer(df_val, datetime_vars)

# Print the first 5 rows of df_val
df_val.head()

,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state,cylinders_unknown,posting_date_year,posting_date_month
0,raleigh / durham / CH,5950,2010.0,honda,odyssey,good,4.0,gas,184713.0,clean,automatic,fwd,van,blue,nc,0,2021,4
1,boise,8495,2011.0,dodge,grand caravan,good,8.0,gas,94164.0,clean,automatic,fwd,mini-van,blue,id,0,2021,4
2,denver,27990,2020.0,ford,transit connect cargo van,good,6.0,other,3772.0,clean,other,fwd,van,white,co,1,2021,5
3,southeast KS,41000,2019.0,mini,t camper van,excellent,4.0,gas,29288.0,clean,automatic,Unknown,van,Unknown,ks,0,2021,4
4,san antonio,16995,2012.0,ford,expedition,Unknown,6.0,other,92248.0,clean,automatic,fwd,SUV,Unknown,tx,1,2021,4


In [56]:
# Call datetime_transformer on df_test
# See the implementation in pmlm_utilities.ipynb
df_test = datetime_transformer(df_test, datetime_vars)

# Print the first 5 rows of df_test
df_test.head()

,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state,cylinders_unknown,posting_date_year,posting_date_month
0,south jersey,16985,2015.0,lincoln,mkc,Unknown,4.0,gas,85933.0,clean,automatic,4wd,SUV,Unknown,nj,1,2021,4
1,cleveland,500,2011.0,jeep,grand cherokee,Unknown,8.0,gas,80125.0,clean,automatic,Unknown,Unknown,Unknown,oh,1,2021,4
2,gold country,1800,1988.0,toyota,pickup,fair,6.0,gas,298000.0,clean,manual,rwd,Unknown,brown,ca,0,2021,4
3,lynchburg,26990,2014.0,chevrolet,silverado 1500 crew,good,4.0,gas,63129.0,clean,automatic,Unknown,pickup,black,va,0,2021,4
4,wyoming,29999,2013.0,gmc,sierra,Unknown,6.0,diesel,229023.0,clean,automatic,4wd,pickup,Unknown,wy,1,2021,4


**Encoding the data**

Combining the training, validation and test data

The code below shows how to combine the training, validation and test data.

In [57]:
# Combine df_train, df_val and df_test
df = pd.concat([df_train, df_val, df_test], sort=False)

# Print the unique data type of variables in df
pd.DataFrame(df.dtypes.unique(), columns=['dtype'])

,dtype
0,object
1,int64
2,float64
3,int32


**Identifying categorical variables**

The code below shows how to find categorical variables (whose data type is dtype) and their number of unique value.

In [58]:
def cat_var_checker(df, dtype='object'):
    """
    The categorical variable checker

    Parameters
    ----------
    df : the dataframe
    dtype : the data type categorical variables should have, 'object' by default
            i.e., if a variable has this data type, it should be a categorical variable

    Returns
    ----------
    The dataframe of categorical variables and their number of unique value
    """

    # Get the dataframe of categorical variables and their number of unique value
    df_cat = pd.DataFrame([[var, df[var].nunique(dropna=False)]
                           # If the data type is dtype
                           for var in df.columns if df[var].dtype == dtype],
                          columns=['var', 'nunique'])

    # Sort df_cat in accending order of the number of unique value
    df_cat = df_cat.sort_values(by='nunique', ascending=False).reset_index(drop=True)

    return df_cat

In [59]:
# Call cat_var_checker on df
# See the implementation in pmlm_utilities.ipynb
df_cat = cat_var_checker(df)

# Print the dataframe
df_cat

,var,nunique
0,model,28936
1,region,404
2,state,51
3,manufacturer,42
4,type,14
5,paint_color,13
6,condition,7
7,title_status,7
8,fuel,5
9,drive,4


**Encoding categorical features**

The code below shows how to encode categorical features in the combined data.

In [60]:
high_card_vars = ['model', 'region']


In [61]:
# One-hot-encode the categorical features in the combined data
df = pd.get_dummies(df, columns=np.setdiff1d(df_cat['var'], high_card_vars + [target]))

# Print the first 5 rows of df
df.head()

,region,price,year,model,cylinders,odometer,cylinders_unknown,posting_date_year,posting_date_month,condition_Unknown,...,type_coupe,type_hatchback,type_mini-van,type_offroad,type_other,type_pickup,type_sedan,type_truck,type_van,type_wagon
0,orange county,11595,2014.0,optima,4.0,115166.0,0,2021,4,True,...,False,False,False,False,False,False,True,False,False,False
1,daytona beach,3100,2004.0,q45,8.0,202.0,0,2021,4,False,...,False,False,False,False,False,False,True,False,False,False
2,san diego,0,2019.0,4runner sr5 suv,6.0,20065.0,1,2021,5,False,...,False,False,False,False,False,False,False,False,False,False
3,billings,12900,2018.0,malibu,4.0,93000.0,0,2021,5,False,...,False,False,False,False,False,False,True,False,False,False
4,tampa bay area,22790,2012.0,f-150 xlt,6.0,98549.0,0,2021,4,False,...,False,False,False,False,False,False,False,True,False,False


**Separating the training, validation and test data**

The code below shows how to separate the training, validation and test data.

In [62]:
# Separating the training data
df_train = df.iloc[:df_train.shape[0], :]

# Separating the validation data
df_val = df.iloc[df_train.shape[0]:df_train.shape[0] + df_val.shape[0], :]

# Separating the test data
df_test = df.iloc[df_train.shape[0] + df_val.shape[0]:, :]

In [63]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.0 MB/s eta 0:00:00


In [64]:
from category_encoders import TargetEncoder

te = TargetEncoder(cols=['model', 'region'])

df_train[['model', 'region']] = te.fit_transform(
    df_train[['model', 'region']],
    df_train[target]
)

df_val[['model', 'region']] = te.transform(df_val[['model', 'region']])
df_test[['model', 'region']] = te.transform(df_test[['model', 'region']])

In [65]:
# Print the dimension of df_train
pd.DataFrame([[df_train.shape[0], df_train.shape[1]]], columns=['# rows', '# columns'])
df_train.head()

,region,price,year,model,cylinders,odometer,cylinders_unknown,posting_date_year,posting_date_month,condition_Unknown,...,type_coupe,type_hatchback,type_mini-van,type_offroad,type_other,type_pickup,type_sedan,type_truck,type_van,type_wagon
0,16846.933923,11595,2014.0,9561.175973,4.0,115166.0,0,2021,4,True,...,False,False,False,False,False,False,True,False,False,False
1,16266.320000,3100,2004.0,44131.829949,8.0,202.0,0,2021,4,False,...,False,False,False,False,False,False,True,False,False,False
2,14121.395020,0,2019.0,49129.312907,6.0,20065.0,1,2021,5,False,...,False,False,False,False,False,False,False,False,False,False
3,23860.345078,12900,2018.0,8014.234002,4.0,93000.0,0,2021,5,False,...,False,False,False,False,False,False,True,False,False,False
4,17647.335878,22790,2012.0,17876.907534,6.0,98549.0,0,2021,4,False,...,False,False,False,False,False,False,False,True,False,False


In [66]:
# Print the dimension of df_val
pd.DataFrame([[df_val.shape[0], df_val.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,83396,155


In [67]:
# Print the dimension of df_test
pd.DataFrame([[df_test.shape[0], df_test.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,83313,155


Feature Engineering

Create car_age

In [68]:
df_train['car_age'] = df_train['posting_date_year'] - df_train['year']
df_val['car_age']   = df_val['posting_date_year'] - df_val['year']
df_test['car_age']  = df_test['posting_date_year'] - df_test['year']

In [69]:
cols_to_drop = ['posting_date_year']

df_train = df_train.drop(columns=cols_to_drop, errors='ignore')
df_val   = df_val.drop(columns=cols_to_drop, errors='ignore')
df_test  = df_test.drop(columns=cols_to_drop, errors='ignore')

**Splitting the feature and target**

The code below shows how to split the feature and target.

In [70]:
# Get the feature matrix
X_train = df_train[np.setdiff1d(df_train.columns, [target])].values
X_val = df_val[np.setdiff1d(df_val.columns, [target])].values
X_test = df_test[np.setdiff1d(df_test.columns, [target])].values

# Get the target vector
y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

**Scaling the data**

**Standardization **

The code below shows how to standardize the data.

In [71]:
from sklearn.preprocessing import StandardScaler

# The StandardScaler
ss = StandardScaler()

**Standardizing the features**

The code below shows how to standardize the features.

In [72]:
# Standardize the training data
X_train = ss.fit_transform(X_train)

# Standardize the validation data
X_val = ss.transform(X_val)

# Standardize the test data
X_test = ss.transform(X_test)

**Standardizing the target**

The code below shows how to standardize the features.

In [73]:
# Standardize the training data
y_train = ss.fit_transform(y_train.reshape(-1, 1)).reshape(-1)

# Standardize the validation data
y_val = ss.transform(y_val.reshape(-1, 1)).reshape(-1)

# Standardize the test data
y_test = ss.transform(y_test.reshape(-1, 1)).reshape(-1)

**Getting the predefined split cross-validator**

In [74]:
from sklearn.model_selection import PredefinedSplit

def get_train_val_ps(X_train, y_train, X_val, y_val):
    """
    Get the:
    feature matrix and target velctor in the combined training and validation data
    target vector in the combined training and validation data
    PredefinedSplit

    Parameters
    ----------
    X_train : the feature matrix in the training data
    y_train : the target vector in the training data
    X_val : the feature matrix in the validation data
    y_val : the target vector in the validation data

    Return
    ----------
    The feature matrix in the combined training and validation data
    The target vector in the combined training and validation data
    PredefinedSplit
    """

    # Combine the feature matrix in the training and validation data
    X_train_val = np.vstack((X_train, X_val))

    # Combine the target vector in the training and validation data
    y_train_val = np.vstack((y_train.reshape(-1, 1), y_val.reshape(-1, 1))).reshape(-1)

    # Get the indices of training and validation data
    train_val_idxs = np.append(np.full(X_train.shape[0], -1), np.full(X_val.shape[0], 0))

    # The PredefinedSplit
    ps = PredefinedSplit(train_val_idxs)

    return X_train_val, y_train_val, ps

Hyperparameter Tuning

In [75]:
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import LinearRegression, Ridge
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

models = {
    'sgd': SGDRegressor(random_state=random_seed),
    'lr': LinearRegression(),
    # 'ridge': Ridge(random_state=random_seed),
    'xgb': XGBRegressor(random_state=random_seed),
    'rf': RandomForestRegressor(random_state=random_seed)
}

**Creating the dictionary of the pipelines**

**In the dictionary:**

the key is the acronym of the model the value is the pipeline, which, for now, only includes the model

In [76]:
from sklearn.pipeline import Pipeline

pipes = {}

for acronym, model in models.items():
    pipes[acronym] = Pipeline([('model', model)])

**Getting the predefined split cross-validator**

In [77]:
# Get the:
# feature matrix and target velctor in the combined training and validation data
# target vector in the combined training and validation data
# PredefinedSplit
# See the implementation in pmlm_utilities.ipynb
X_train_val, y_train_val, ps = get_train_val_ps(X_train, y_train, X_val, y_val)

In [78]:
param_grids = {}

The parameter grid for SGDRegressor

The hyperparameters we want to fine-tune are:

eta0

alpha


In [79]:
# The parameter grid of eta
eta_grid = [0.001, 0.02]

# The parameter grid of alpha
alpha_grid = [0.01, 0.1]

# Update param_grids
param_grids['sgd'] = [{'model__eta0': eta_grid,
                       'model__alpha': alpha_grid}]

The parameter grid for LinearRegression_MBGD

The hyperparameters we want to fine-tune are:

eta

alpha



In [80]:
# The parameter grid of eta
eta_grid = [0.001, 0.02]

# The parameter grid of alpha
alpha_grid = [0.01, 0.1]

# Update param_grids
param_grids['lr'] = [{}]

Param grids for Random Forest

In [81]:
param_grids['rf'] = [{
    'model__n_estimators': [100],
    'model__max_depth': [10]
}]



Param grids fro XGB

In [82]:
param_grids['xgb'] = [{
    'model__n_estimators': [100],
    'model__max_depth': [3],
    'model__learning_rate': [0.1]
}]

In [83]:
# Make directory
directory = os.path.join(abspath_curr, 'result', 'mnist', 'cv_results', 'GridSearchCV')
os.makedirs(directory, exist_ok=True)


**Tuning the hyperparameters**

In [84]:
from sklearn.model_selection import GridSearchCV

# The list of [best_score_, best_params_, best_estimator_] obtained by GridSearchCV
best_score_params_estimator_gs = []

# For each model
for acronym in pipes.keys():
    # GridSearchCV
    gs = GridSearchCV(estimator=pipes[acronym],
                      param_grid=param_grids[acronym],
                      scoring='neg_mean_squared_error',
                      n_jobs=2,
                      cv=ps,
                      return_train_score=True)

    # Fit the pipeline
    gs = gs.fit(X_train_val, y_train_val)

    # Update best_score_params_estimator_gs
    best_score_params_estimator_gs.append([gs.best_score_, gs.best_params_, gs.best_estimator_])

    # Sort cv_results in ascending order of 'rank_test_score' and 'std_test_score'
    cv_results = pd.DataFrame.from_dict(gs.cv_results_).sort_values(by=['rank_test_score', 'std_test_score'])

    # Get the important columns in cv_results
    important_columns = ['rank_test_score',
                         'mean_test_score',
                         'std_test_score',
                         'mean_train_score',
                         'std_train_score',
                         'mean_fit_time',
                         'std_fit_time',
                         'mean_score_time',
                         'std_score_time']

    # Move the important columns ahead
    cv_results = cv_results[important_columns + sorted(list(set(cv_results.columns) - set(important_columns)))]

    # Write cv_results file
    cv_results.to_csv(path_or_buf=os.path.join(directory, f'{acronym}.csv'), index=False)

# Sort best_score_params_estimator_gs in descending order of the best_score_
best_score_params_estimator_gs = sorted(best_score_params_estimator_gs, key=lambda x: x[0], reverse=True)

# Print best_score_params_estimator_gs
pd.DataFrame(best_score_params_estimator_gs, columns=['best_score', 'best_param', 'best_estimator'])


,best_score,best_param,best_estimator
0,-1.590088,"{'model__alpha': 0.1, 'model__eta0': 0.001}","(SGDRegressor(alpha=0.1, eta0=0.001, random_st..."
1,-1.603457,{},(LinearRegression())
2,-1.712839,"{'model__learning_rate': 0.1, 'model__max_dept...","(XGBRegressor(base_score=None, booster=None, c..."
3,-1.728086,"{'model__max_depth': 10, 'model__n_estimators'...","((DecisionTreeRegressor(max_depth=10, max_feat..."


In [85]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

best_model = best_score_params_estimator_gs[0][2]
pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 2))


MAE: 0.01
RMSE: 1.1
R2: -0.0


**Important Features**


**Get a fitted linear model's coefficients**


In [86]:
coef_model = next(
    estimator for _, _, estimator in best_score_params_estimator_gs
    if hasattr(estimator.named_steps['model'], 'coef_')
)
coef_model_name = type(coef_model.named_steps['model']).__name__
coefs = coef_model.named_steps['model'].coef_
print(f'Using coefficients from {coef_model_name}.')


Using coefficients from SGDRegressor.


**Get the important features**

In [87]:
import pandas as pd

feature_names = np.setdiff1d(df_train.columns, [target])

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs
})

coef_df['importance'] = coef_df['coefficient'].abs()

coef_df = coef_df.sort_values(by='importance', ascending=False)

coef_df.head(10)

,feature,coefficient,importance
61,model,0.008046,0.008046
77,region,0.002104,0.002104
148,type_pickup,0.001414,0.001414
120,state_tn,-0.001220,0.001220
109,state_nj,-0.001158,0.001158
25,manufacturer_buick,0.001084,0.001084
91,state_id,0.001012,0.001012
19,manufacturer_Unknown,0.000861,0.000861
100,state_mi,0.000842,0.000842
14,fuel_diesel,0.000707,0.000707


**engine_cc**

Cars with larger engine sizes tend to have higher prices, indicating that performance is a key factor in valuation.

**make_year**

Newer cars generally have higher prices, showing that vehicle age strongly affects depreciation.

**fuel_type_Electric**

Electric vehicles are associated with higher prices, likely due to newer technology and higher market demand.

**owner_count**

Cars with more previous owners tend to have lower prices, suggesting concerns about condition and reliability.

**mileage_kmpl**

Vehicles with better fuel efficiency tend to be priced higher, reflecting consumer preference for economical cars.

**fuel_type_Petrol / Diesel**

Petrol and diesel cars tend to be priced lower than electric vehicles, showing a clear price difference across fuel types.

**brand_BMW / brand_Volkswagen**

Brand has a minor positive effect on price, but its influence is relatively small compared to technical features.

**color_Red**

Color has very little impact on price, suggesting it is not an important factor in valuation.

## **Conclusions**

The model shows strong performance with an R2 of 0.85, indicating that it explains 85% of the variation in car prices. The MAE of 0.31 suggests that the average prediction error is relatively small. The RMSE of 0.39 is slightly higher than the MAE, indicating the presence of some larger errors, but overall the model provides reliable predictions.

The results show that car prices are mainly influenced by technical and usage-related features. Engine size and vehicle age have the strongest impact, with larger and newer cars being more expensive. Electric vehicles also tend to have higher prices, while cars with more previous owners are cheaper. Fuel efficiency has a positive effect, whereas petrol and diesel cars are generally priced lower than electric vehicles. In contrast, brand and color have only minor influence on price.


## **References**

1. Rohith Anand. *Used Car Price Prediction* dataset. Kaggle. https://www.kaggle.com/datasets/therohithanand/used-car-price-prediction

2. Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., and Duchesnay, E. (2011). *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research, 12, 2825-2830. https://jmlr.org/papers/v12/pedregosa11a.html

3. Chen, T. and Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining. https://doi.org/10.1145/2939672.2939785

4. Breiman, L. (2001). *Random Forests*. Machine Learning, 45, 5-32. https://doi.org/10.1023/A:1010933404324

5. Abadi, M., Agarwal, A., Barham, P., Brevdo, E., Chen, Z., Citro, C., Corrado, G. S., Davis, A., Dean, J., Devin, M., Ghemawat, S., Goodfellow, I., Harp, A., Irving, G., Isard, M., Jia, Y., Jozefowicz, R., Kaiser, L., Kudlur, M., Levenberg, J., Mane, D., Monga, R., Moore, S., Murray, D., Olah, C., Schuster, M., Shlens, J., Steiner, B., Sutskever, I., Talwar, K., Tucker, P., Vanhoucke, V., Vasudevan, V., Viegas, F., Vinyals, O., Warden, P., Wattenberg, M., Wicke, M., Yu, Y., and Zheng, X. (2015). *TensorFlow: Large-Scale Machine Learning on Heterogeneous Distributed Systems*. https://download.tensorflow.org/paper/whitepaper2015.pdf
